<a href="https://colab.research.google.com/github/oliviaberset/2026_solute_transport/blob/main/Ex4_agricultural_products.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing the transport of chloride and nitrate in an agricultural catchment

We will explore how the water quality in an agricultural catchment evolves throughout the year. We will explore real-world data on nitrate and chloride concentrations, and evaluate the environmental factors influencing nitrate removal.

## Context

You are tasked with analyzing stream water quality data from the Hupsel Brook catchment, which is a lowland agricultural site in the Netherlands.

<figure style="max-width: 800px; width: 100%;">
  <img src="Hupsel_map.jpg"
       alt="Map and pictures of the Hupsel Brook catchment"
       style="width: 90%; height: auto;" />
  <figcaption style="width: 90%;">
    Figure 1: Maps (a-b) of the Hupsel Brook catchment (6.5 km<sup>2</sup>) and pictures of the outlet (c) and landscape (d), highlighting the lowland agricultural characteristics of the site. Photo credit: Y. van der Velde.
  </figcaption>
</figure>

This site is interesting because it underwent a strong reduction in the use of agricultural fertilizers, which resulted in a strong decrease in nutrients in streamwater (Figure 2).


<figure style="max-width: 800px; width: 100%;">
  <img src="input_loads_and_streamwater_chemistry.png"
       alt="Input loads and streamwater concentrations"
       style="width: 90%; height: auto;" />
  <figcaption style="width: 90%;">
    Figure 2: Timeseries of estimated chloride and nitrogen input loads to the catchment, which decreased substantially over the years (grey line, right axis). Due to the decreased input loads, streamwater concentrations for these elements decreased as well, while maintaining important seasonal fluctuations (red dots, left axis).
  </figcaption>
</figure>

The key solutes in this dataset are chloride, nitrogen, and phosphorus. In the Hupsel Brook catchment, all three originate primarily from manure applications, but they follow different transport pathways and undergo distinct sorption and transformation processes. Chloride is relatively inert and interacts minimally with soils or vegetation. Nitrogen, by contrast, is a nutrient that is preferentially taken up by vegetation and is also converted to nitrogen gas through denitrification. Both chloride and nitrogen are generally assumed to move freely through the subsurface, experiencing little sorption or retardation. Phosphorus behaves very differently: it strongly sorbs to soils and is mainly mobilized during high‑flow events, either via overland flow or attached to suspended sediments. It is also a nutrient and therefore likely subject to preferential plant uptake, but unlike nitrogen, it does not undergo gaseous loss.

Comprehensive information about the catchment, its measurements and history can be found in the paper by Brauer et al. ([2018](https://doi.org/10.2136/vzj2018.03.0056))

## The data

A compiled hydrochemical dataset for the Hupsel Brook catchment has been recently published by van der Velde et al., ([2026](https://doi.org/10.5281/zenodo.19237432)), including more than 2,200 individual samples analyzed for a range of chemical and physical properties.


You will find the file **WaterQuality_combined_1983-2008.csv** in the **data** folder.

### Table: `WaterQuality_combined_1983-2008.csv`  

This table includes measurements of chloride, N-constituents, P-consituents and occasional other parameters, spanning the period 1983-2008 (with gaps).

Missing data appear as empty cells and timestamps are in UTC+01 (CET)


| Column               | Type     | Units | Description                                                       |
|----------------------|----------|-------|-------------------------------------------------------------------|
| `AvgDate`            | Datetime |      | Average time of collection, format YYYY-MM-DD HH:mm:SS, time zone UTC+01 |
| `StartDate`          | Datetime |      | Start time of sample collection, format YYYY-MM-DD HH:mm:SS, time zone UTC+01 |
| `EndDate`            | Datetime |      | End time of sample collection, format YYYY-MM-DD HH:mm:SS, time zone UTC+01 |
| `TimeReconstructed`  | Boolean  |      | Flag indicating whether the time of sampling was reconstructed    |
| `Year`               | Integer  |      | Year of sample collection                                         |
| `Month`              | Integer  |      | Month of sample collection                                        |
| `DOY`                | Integer  |      | Day of the year of sample collection                              |
| `Cl`                 | Float    | mg/l  | Chloride concentration                                            |
| `N-NO3`              | Float    | mg/l  | Nitrate nitrogen concentration                                    |
| `N-NO2`              | Float    | mg/l  | Nitrite nitrogen concentration                                    |
| `N-NH4`              | Float    | mg/l  | Ammonium nitrogen concentration                                   |
| `P-PO4`              | Float    | mg/l  | Phosphate phosphorus concentration                                |
| `P-tot`              | Float    | mg/l  | Total phosphorus concentration                                    |
| `NO3`                | Float    | mg/l  | Nitrate concentration                                             |
| `pH`                 | Float    |      | pH                                                                |
| `EC`                 | Float    | μS/cm | Electrical conductivity                                           |
| `Source`             | String   |      | Sample dataset source                                             |
| `mean_Q`      | Float    | mm/h  | Average discharge between StartDate and EndDate                   |
| `mean_Q_log10`       | Float    | log₁₀(mm/h) | log10 of the average discharge                              |

---



## Exercise

This notebook proposes two activities to learn about transport of agricultural products

- Activity 1: Temporal concentration dynamics along the year
- Activity 2: Processes revealed by C-Q relationships

### Load the data

In [ ]:
# Load the main packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
# Define the file path (you may have a different one)
filepath = os.path.join('data', 'WaterQuality_combined_1983-2008.csv')

# Load the data into a pandas dataframe
df_all = pd.read_csv(filepath)

# Convert the 'date' column to a datetime format
df_all['AvgDate'] = pd.to_datetime(df_all['AvgDate'])

# Show a preview of the different columns and their data types
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2373 entries, 0 to 2372
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   AvgDate            2373 non-null   datetime64[ns]
 1   StartDate          2373 non-null   object        
 2   EndDate            2373 non-null   object        
 3   TimeReconstructed  2373 non-null   bool          
 4   Year               2373 non-null   int64         
 5   Month              2373 non-null   int64         
 6   DOY                2373 non-null   int64         
 7   Cl                 2364 non-null   float64       
 8   N-NO3              2369 non-null   float64       
 9   N-NO2              2297 non-null   float64       
 10  N-NH4              2344 non-null   float64       
 11  P-PO4              2173 non-null   float64       
 12  P-tot              2294 non-null   float64       
 13  NO3                2369 non-null   float64       
 14  pH      

---

### Activity 1: problem formulation

The manure application period is usually concentrated between March and October. What is the evolution of solute concentration over a typical year? If you worked for an agency that has to estimate the highest nutrient concentrations as a similar catchment, when would you go and sample the stream?

### Activity 1: development

Suggested strategy:
- Focus on some of the early years between 1983-1994 when solute data was collected more frequently
- Make timeseries plots of the key agricultural products that allow you to draw conclusions

### Activity 1: Discussion and Conclusions



---

### Activity 2: problem formulation

All agricultural products have the same primary source (manure) but their solute concentration dynamics in streamwater are rather different. What visualizations can best help you see the differences among these solutes? What processes are revealed by such visualizations?

### Activity 2: development

Suggested strategy:
- You are free to come up with your own visualizations. To compare solutes directly, it is useful to normalize their concentrations. A typical normalization is obtained by subtracting the mean and dividing by the standard deviation.
- One thing to pay attention to: it is nice to use all the available data, but you need to consider that these solutes have a long-term decreasing trend that may disturb the visualizations. In such a case, it can be important to de-trend the series. There are many ways to do this. The simplest approach is normalizing by each year's mean and standard deviation. You are welcome to explore other strategies.

### Activity 2: Discussion and Conclusions


